# Ensemble Híbrido Guiado por Regime (QSVM → Random Forest)

Combina os dois componentes que demonstraram valor no estudo: a **classificação de regime pelo QSVM** (kernel quântico) e a **previsão pelo Random Forest** (melhor clássico). Para cada semana, o classificador quântico identifica o regime epidemiológico (declínio, endemia ou crescimento) e roteia a amostra a um Random Forest especializado naquele regime. Diferentemente do antigo roteador dinâmico, o roteamento usa uma classificação legítima (sem rótulos-oráculo), evitando vazamento. Kernel: `qml_dengue`.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import os, json, sys, time
import numpy as np
import matplotlib.pyplot as plt
import pennylane as qml
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, os.path.join(REPO_ROOT, "src"))
sys.path.insert(0, os.path.abspath(".."))
from feature_engineering import construir_features, splits_validacao
from utils_qml import metricas, salvar_padrao, plot_pred, validar_json_saida

CACHE = os.path.join(REPO_ROOT, "data", "dados_dengue_df_real.json")
with open(CACHE, encoding="utf-8") as f:
    dados_brutos = json.load(f)
dataset = construir_features(dados_brutos, n_lags=4)
splits  = splits_validacao(dataset)

NOMES = {0: ("C1", "Transmissão normal/crescente  (out/2023 – out/2024)"),
         1: ("C2", "Pico recorde 25.714 casos/sem  (jun/2024 – jun/2025)"),
         2: ("C3", "Pós-surto, Rt < 1              (out/2024 – jun/2025)")}
CENARIOS = {}
for idx, split in enumerate(splits[:3]):
    nome, desc = NOMES[idx]
    tr, te = split["treino"], split["teste"]
    CENARIOS[nome] = {"X_train": np.array(tr["X"]), "y_train": np.array(tr["y"]),
                      "X_test": np.array(te["X"]), "y_test": np.array(te["y"]), "nome": desc}

CONFIG   = {"n_qubits": 6, "n_layers_kernel": 2, "seed": 42, "C_svm": 10.0}
FEAT_IDX = [0, 1, 2, 3, 4, 5, 6, 7]   # casos_lag1..4 + Rt_lag1/2 + p_rt1 + receptivo
RT_IDX   = 4                          # Rt_lag1 (para rotular regime)
print("Cenários:", {c: (len(d['X_train']), len(d['X_test'])) for c, d in CENARIOS.items()})

Cenários: {'C1': (36, 143), 'C2': (88, 91), 'C3': (125, 54)}


In [2]:
# ── Kernel quântico (ZZ feature map, Havlíček 2019) + rótulos de regime ──
n_q = CONFIG["n_qubits"]
dev_k = qml.device("default.qubit", wires=n_q)

@qml.qnode(dev_k)
def feature_map(x):
    x_pad = x[:n_q] if len(x) >= n_q else np.pad(x, (0, n_q - len(x)))
    for _ in range(CONFIG["n_layers_kernel"]):
        for i in range(n_q):
            qml.Hadamard(wires=i)
            qml.RZ(x_pad[i % len(x_pad)], wires=i)
        for i in range(n_q - 1):
            qml.CZ(wires=[i, i+1])
            fi, fj = x_pad[i % len(x_pad)], x_pad[(i+1) % len(x_pad)]
            qml.RZ((np.pi - fi) * (np.pi - fj), wires=i)
            qml.RZ((np.pi - fi) * (np.pi - fj), wires=i+1)
    return qml.state()

def quantum_kernel(x1, x2):
    s1, s2 = feature_map(x1), feature_map(x2)
    return float(np.abs(np.dot(np.conj(s1), s2)) ** 2)

def build_kernel_matrix(X_a, X_b):
    K = np.zeros((len(X_a), len(X_b)))
    for i in range(len(X_a)):
        for j in range(len(X_b)):
            K[i, j] = quantum_kernel(X_a[i], X_b[j])
    return K

def rotular_regime(X):
    rt = X[:, RT_IDX]
    lab = np.ones(len(rt), dtype=int)   # 1 = endemia
    lab[rt < 0.85] = 0                  # declínio
    lab[rt > 1.20] = 2                  # crescimento
    return lab

print("Kernel quântico e rótulos de regime prontos.")

Kernel quântico e rótulos de regime prontos.


In [3]:
# ── Classificação de regime (QSVM) + Random Forest por regime (LENTO: ~4-5 min) ──
LABEL = {0: "Declínio", 1: "Endemia", 2: "Crescimento"}
N_BOOT = 5
np.random.seed(42)
RESULTADOS = {}

def novo_rf(seed):
    return RandomForestRegressor(n_estimators=200, max_depth=12, min_samples_leaf=3,
                                 random_state=seed, n_jobs=-1)

for cen, d in CENARIOS.items():
    t0 = time.time()
    Xtr, ytr = d["X_train"], d["y_train"]
    Xte, yte = d["X_test"],  d["y_test"]
    reg_tr = rotular_regime(Xtr)

    # 1) classificação de regime com kernel quântico
    sc = MinMaxScaler(feature_range=(0.01, np.pi))
    Ktr_f = sc.fit_transform(Xtr[:, FEAT_IDX])
    Kte_f = sc.transform(Xte[:, FEAT_IDX])
    K_train = build_kernel_matrix(Ktr_f, Ktr_f)
    K_test  = build_kernel_matrix(Kte_f, Ktr_f)
    clf = SVC(kernel="precomputed", C=CONFIG["C_svm"], random_state=42)
    clf.fit(K_train, reg_tr)
    reg_te = clf.predict(K_test)
    acc = accuracy_score(rotular_regime(Xte), reg_te)
    dist = {LABEL[r]: int((reg_te == r).sum()) for r in [0, 1, 2]}

    # 2) Random Forest especializado por regime + bootstrap; roteia pelo regime previsto
    preds = np.zeros((N_BOOT, len(Xte)))
    for b in range(N_BOOT):
        rng = np.random.RandomState(42 + b)
        rfs = {}
        for r in [0, 1, 2]:
            m = np.where(reg_tr == r)[0]
            if len(m) >= 3:
                idx = rng.choice(m, len(m), replace=True)
                rf = novo_rf(42 + b); rf.fit(Xtr[idx], np.log1p(ytr[idx])); rfs[r] = rf
        idxg = rng.choice(len(Xtr), len(Xtr), replace=True)
        rf_glob = novo_rf(42 + b); rf_glob.fit(Xtr[idxg], np.log1p(ytr[idxg]))
        for i in range(len(Xte)):
            model = rfs.get(int(reg_te[i]), rf_glob)
            preds[b, i] = max(np.expm1(float(model.predict(Xte[i:i+1])[0])), 0.0)

    med = np.median(preds, axis=0)
    mtr = metricas(yte, med, preds, nome=f"Ensemble_{cen}")
    RESULTADOS[cen] = {**mtr, "preds_matrix": preds, "mediana": med, "y_test": yte,
                       "acc_regime": float(acc), "tempo_s": time.time() - t0}
    print(f"{cen}: R2={mtr['R2']:.4f} | WIS={mtr['WIS']:.2f} | acc_regime={acc:.3f} | regimes_teste={dist} | {RESULTADOS[cen]['tempo_s']:.0f}s")

C1: R2=0.1598 | WIS=1789.91 | acc_regime=0.860 | regimes_teste={'Declínio': 142, 'Endemia': 1, 'Crescimento': 0} | 117s
C2: R2=0.0944 | WIS=2693.44 | acc_regime=0.824 | regimes_teste={'Declínio': 87, 'Endemia': 4, 'Crescimento': 0} | 276s
C3: R2=-9.7337 | WIS=226.92 | acc_regime=0.889 | regimes_teste={'Declínio': 48, 'Endemia': 5, 'Crescimento': 1} | 380s


In [4]:
# ── Tabela + figura ──
print(f"\n{'='*70}")
print(f"{'ENSEMBLE HÍBRIDO GUIADO POR REGIME (QSVM → RF)':^70}")
print(f"{'='*70}")
print(f"{'Cenário':<10}{'R²':>10}{'WIS':>12}{'Acc.Regime':>14}")
print("-"*70)
for cen, r in RESULTADOS.items():
    print(f"{cen:<10}{r['R2']:>10.4f}{r['WIS']:>12.2f}{r['acc_regime']:>14.3f}")
print("="*70)

plot_pred(RESULTADOS,
          "Ensemble Híbrido (regime QSVM → RF): Predição vs. Observado (DF 2022-2025)",
          "ensemble_regime_pred_vs_obs.png")


            ENSEMBLE HÍBRIDO GUIADO POR REGIME (QSVM → RF)            
Cenário           R²         WIS    Acc.Regime
----------------------------------------------------------------------
C1            0.1598     1789.91         0.860
C2            0.0944     2693.44         0.824
C3           -9.7337      226.92         0.889
[SALVO] ensemble_regime_pred_vs_obs.png


In [5]:
SCHEMA_INFO = {
    "algoritmo": "Ensemble_Regime_QSVM_RF",
    "fase": 40,
    "tipo": "hibrido",
    "n_parametros_quanticos": 0,
    "config": {**CONFIG, "roteador": "QSVM (kernel quantico)", "especialista": "RandomForest por regime",
               "n_bootstrap": N_BOOT},
}
doc = salvar_padrao(RESULTADOS, SCHEMA_INFO)
validar_json_saida(doc, contexto="Ensemble_Regime_QSVM_RF")

[PADRAO] fase40_ensemble_regime_qsvm_rf_resultados.json
  Algoritmo : Ensemble_Regime_QSVM_RF
  Tipo      : hibrido
  Parametros quanticos: 0
  C1: R2=+0.1598 | WIS=1789.91 | 116.6s
  C2: R2=+0.0944 | WIS=2693.44 | 275.9s
  C3: R2=-9.7337 | WIS=226.92 | 379.7s
[CONTRATO OK] [Ensemble_Regime_QSVM_RF] JSON valido — todos os campos e invariantes corretos


True